# Developmental InterpretabilityHow do circuits form during training? This module analyzes model checkpoints across training to understand when and how interpretable structures crystallize.Key questions:- When do specific circuits "turn on" during training?- Are there phase transitions where the model restructures internally?- What is the learning order of different capabilities?

## Why This Matters

Developmental interpretability studies how circuits crystallize during training — when do specific heads specialize, when do induction circuits form, and what triggers phase transitions? This connects training dynamics to the static circuits found in fully-trained models.

**Key references:**
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jaximport jax.numpy as jnpimport numpy as npfrom irtk import HookedTransformerConfig, HookedTransformercfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)model = HookedTransformer(cfg)key = jax.random.PRNGKey(42)leaves, treedef = jax.tree.flatten(model)new_leaves = []for leaf in leaves:    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:        key, sk = jax.random.split(key)        new_leaves.append(jax.random.normal(sk, leaf.shape, dtype=leaf.dtype) * 0.1)    else:        new_leaves.append(leaf)model = jax.tree.unflatten(treedef, new_leaves)tokens = jnp.array([0, 1, 2, 3, 4, 5, 6, 7])def metric(logits):    return float(logits[-1, 0])

In [ ]:
from irtk.developmental_interpretability import (    detect_phase_transitions,    track_circuit_formation,    measure_representation_crystallization,    compare_learning_order,    grokking_detector,)

In [ ]:
# Create "checkpoints" (different random models simulating training)checkpoints = []for seed in range(5):    key = jax.random.PRNGKey(seed * 100)    leaves, treedef = jax.tree.flatten(model)    new_leaves = []    for leaf in leaves:        if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:            key, sk = jax.random.split(key)            new_leaves.append(jax.random.normal(sk, leaf.shape, dtype=leaf.dtype) * (0.1 + seed * 0.02))        else:            new_leaves.append(leaf)    checkpoints.append(jax.tree.unflatten(treedef, new_leaves))print(f"Created {len(checkpoints)} checkpoints")

## Phase Transition DetectionCompare representation similarity between consecutive checkpoints to find discontinuous jumps indicating qualitative reorganization.

In [ ]:
result = detect_phase_transitions(checkpoints, tokens)print(f"Similarity curve: {[f'{s:.4f}' for s in result['similarity_curve']]}")print(f"Transition points: {result['transition_points']}")print(f"Smoothness: {result['smoothness']:.4f}")

## Circuit Formation TrackingWhen does a specific circuit become functional? Ablate the circuit at each checkpoint and measure the effect.

In [ ]:
hooks = [("blocks.0.attn.hook_result", lambda x, n: jnp.zeros_like(x))]result = track_circuit_formation(checkpoints, tokens, hooks, metric)print(f"Circuit effects per checkpoint: {[f'{e:.6f}' for e in result['circuit_effects']]}")print(f"Formation checkpoint: {result['formation_checkpoint']}")print(f"Formation curve: {[f'{f:.4f}' for f in result['formation_curve']]}")

## Representation CrystallizationTrack how representation structure changes across training. Lower effective rank means more structured, crystallized representations.

In [ ]:
result = measure_representation_crystallization(checkpoints, tokens)for i, (r, c) in enumerate(zip(result['effective_ranks'], result['condition_numbers'])):    print(f"  Checkpoint {i}: rank={r:.2f}, condition={c:.2f}")print(f"Crystallization index: {result['crystallization_index']:.4f}")

## Learning Order ComparisonWhich circuits form first? Compare the temporal ordering of different capabilities.

In [ ]:
specs = {    "attn_L0": [("blocks.0.attn.hook_result", lambda x, n: jnp.zeros_like(x))],    "attn_L1": [("blocks.1.attn.hook_result", lambda x, n: jnp.zeros_like(x))],}result = compare_learning_order(checkpoints, tokens, specs, metric)print(f"Formation order: {result['formation_order']}")print(f"Earliest circuit: {result['earliest_circuit']}")print(f"Latest circuit: {result['latest_circuit']}")

## Grokking DetectionDetect delayed generalization — when the model memorizes training data but only generalizes much later in training.

In [ ]:
train_tokens = jnp.array([0, 1, 2, 3])test_tokens = jnp.array([10, 11, 12, 13])result = grokking_detector(checkpoints, train_tokens, test_tokens, metric)print(f"Grokking detected: {result['grokking_detected']}")for i in range(len(checkpoints)):    print(f"  CP {i}: train={result['train_metrics'][i]:.4f}, "          f"test={result['test_metrics'][i]:.4f}, "          f"gap={result['generalization_gap'][i]:.4f}, "          f"wnorm={result['weight_norms'][i]:.4f}")